In [2]:
import openeo
import os

import rasterio
from rasterio.mask import mask
from rasterio.merge import merge

import geopandas as gpd

from openeo.processes import if_, is_nodata

from collections import defaultdict

from pathlib import Path

In [3]:
YEARS = [2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023 ,2024, 2025, 2026]

BASE = Path("/mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines")
OUT = BASE / "tests/coherence/"

In [8]:
for year in YEARS:

    in_folder = OUT / str(year)

    # Find all coherence files for each swadth
    files = list(in_folder.glob("sar_coherence_T*.tiff"))

    # Extract the track number from the filename
    groups = defaultdict(list)

    for f in files:
        parts = f.stem.split("_")
        
        track = parts[2]  
        
        groups[track].append(f)

    # Create one composite for each track
    for track, filenames in groups.items():

        print(f"{track}: {len(filenames)} files")

        datasets = [rasterio.open(f) for f in filenames]

        mosaic, transform = merge(
            datasets,
            method="max",
            nodata=0
        )

        profile = datasets[0].profile.copy()
        profile.update(
            height=mosaic.shape[1],
            width=mosaic.shape[2],
            transform=transform,
            count=1,
            nodata=0,
            dtype=mosaic.dtype
        )

        output_file = in_folder / f"sar_coherence_{track}_composite.tiff"

        with rasterio.open(output_file, "w", **profile) as dst:
            dst.write(mosaic[0], 1)

        for src in datasets:
            src.close()

        print(f"{output_file}")

T95: 5 files
/mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/coherence/2016/sar_coherence_T95_composite.tiff
T168: 7 files
/mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/coherence/2016/sar_coherence_T168_composite.tiff
T15: 5 files
/mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/coherence/2016/sar_coherence_T15_composite.tiff
T117: 9 files
/mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/coherence/2016/sar_coherence_T117_composite.tiff
T44: 3 files
/mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/coherence/2016/sar_coherence_T44_composite.tiff
T95: 5 files
/mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/coherence/2017/sar_coherence_T95_composite.tiff
T168: 7 files
/mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/coherence/2017/sar_coherence_T168_composite.tiff
T15: 5 files
/mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/coherence/2017/sar_coherence_T15_composite.tiff
T117: 6 files
/mnt/CEPH_PROJECTS/p

In [4]:
def merge_swaths(files, output_filename):

    datasets = [rasterio.open(f) for f in files]

    mosaic, transform = merge(
        datasets,
        method="max",
        nodata=0
    )

    meta = datasets[0].meta.copy()

    meta.update({
        "height": mosaic.shape[1],
        "width": mosaic.shape[2],
        "transform": transform,
        "count": 1,
        "nodata": 0
    })

    with rasterio.open(output_filename, "w", **meta) as dst:
        dst.write(mosaic)

    for ds in datasets:
        ds.close()

    print(f"Saved: {output_filename}")

In [6]:
import re

ascending_orbits = [15, 44, 117]
descending_orbits = [95, 168]

for year in YEARS:
    in_folder = OUT / str(year) 
    files = list(in_folder.glob("sar_coherence_T*composite.tiff"))

    # Separate files by orbit direction
    ascending_files = []
    descending_files = []

    for f in files:
        # Extract T number from filename
        match = re.search(r"T(\d+)", f.name)
        if match:
            orbit = int(match.group(1))
            if orbit in ascending_orbits:
                ascending_files.append(f)
            elif orbit in descending_orbits:
                descending_files.append(f)

    print(f"Year {year}")
    print("Ascending:")
    for f in ascending_files:
        print(f)

    print("\nDescending:")
    for f in descending_files:
        print(f)

    ascending_filename = in_folder / f"coh_mosaic_ascending_{year}.tiff"
    descending_filename = in_folder / f"coh_mosaic_descending_{year}.tiff"

    merge_swaths(ascending_files, ascending_filename)
    merge_swaths(descending_files, descending_filename)

Year 2016
Ascending:
/mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/coherence/2016/sar_coherence_T15_composite.tiff
/mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/coherence/2016/sar_coherence_T117_composite.tiff
/mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/coherence/2016/sar_coherence_T44_composite.tiff

Descending:
/mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/coherence/2016/sar_coherence_T95_composite.tiff
/mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/coherence/2016/sar_coherence_T168_composite.tiff
Saved: /mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/coherence/2016/coh_mosaic_ascending_2016.tiff
Saved: /mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/coherence/2016/coh_mosaic_descending_2016.tiff
Year 2017
Ascending:
/mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/coherence/2017/sar_coherence_T15_composite.tiff
/mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/coherence/2017/sa